# 01 - Coded Chat Data Loading

This notebook parses the existing coded conversations and creates a flat, grouped Stage 1 dataset. The predator posts are the only messages retained for the 200/600/900 classifier; uncoded victim posts are excluded rather than assigned a guessed label. Existing XML codes are used directly, so no manual labeling is performed.

All data paths are relative to this notebook's workspace root.

In [ ]:
from pathlib import Path
from collections import Counter
import re
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

WORKSPACE = Path.cwd()
CODED_CHAT_DIR = WORKSPACE / "encoded_chats_xml"
OUTPUT_DIR = WORKSPACE / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## XML parsing policy

The coded XML files can contain a local DTD declaration and escaped angle brackets. The parser below removes only the DOCTYPE declaration and protects bare ampersands before XML parsing. It does not fabricate codes, timestamps, authors, or victim labels.

In [ ]:
DOCTYPE_PATTERN = re.compile(r"<!DOCTYPE[^>]*>", flags=re.IGNORECASE | re.DOTALL)
BARE_AMPERSAND_PATTERN = re.compile(r"&(?!#\d+;|#x[0-9A-Fa-f]+;|[A-Za-z_][A-Za-z0-9_.-]*;)")


def parse_coded_xml(path: Path) -> tuple[ET.Element, str]:
    raw = path.read_text(encoding="utf-8-sig")
    raw = DOCTYPE_PATTERN.sub("", raw)
    raw = BARE_AMPERSAND_PATTERN.sub("&amp;", raw)
    return ET.fromstring(raw), raw


def text_at(root: ET.Element, path: str) -> str:
    node = root.find(path)
    return "" if node is None or node.text is None else node.text.strip()


def load_coded_chats(chat_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    diagnostics = []

    xml_files = sorted(chat_dir.glob("*.xml"))
    for chat_path in xml_files:
        root, raw_xml = parse_coded_xml(chat_path)
        predator = text_at(root, "./PREDATOR/SCREENNAME/USERNAME")
        victim = text_at(root, "./VICTIM/SCREENNAME/USERNAME")
        post_nodes = list(root.findall("./POST"))

        coded_count = 0
        uncoded_count = 0
        retained_categories = []

        source_idx = 1
        for post in post_nodes:
            username = text_at(post, "./USERNAME")
            body = text_at(post, "./BODY")
            datetime_text = text_at(post, "./DATETIME")
            category = text_at(post, "./CODING/CATEGORY")

            if category:
                coded_count += 1
            else:
                uncoded_count += 1

            if username == predator and category in {"200", "600", "900"}:
                retained_categories.append(category)
                rows.append({
                    "chat_id": chat_path.stem,
                    "msg_idx": source_idx,
                    "username": username,
                    "datetime_text": datetime_text or np.nan,
                    "text": body,
                    "category": category,
                    "speaker_role": "predator",
                })

            source_idx += 1

        categories = Counter(retained_categories)
        diagnostics.append({
            "chat_id": chat_path.stem,
            "post_count": len(post_nodes),
            "coded_post_count": coded_count,
            "uncoded_post_count": uncoded_count,
            "retained_stage_count": sum(categories.values()),
            "count_200": categories["200"],
            "count_600": categories["600"],
            "count_900": categories["900"],
            "predator_username": predator,
            "victim_username": victim,
        })

    columns = [
        "chat_id", "msg_idx", "username", "datetime_text", "text",
        "category", "speaker_role"
    ]
    messages = pd.DataFrame(rows, columns=columns)
    chat_summary = pd.DataFrame(diagnostics)
    return messages, chat_summary


messages, chat_summary = load_coded_chats(CODED_CHAT_DIR)
print(f"Found {chat_summary.shape[0]} coded chat files")
print(f"Retained {messages.shape[0]} predator-coded messages")

## Validation and grouped sequence diagnostics

Each row is checked against the three known stages. Stage transitions are reported as an informational sequence diagnostic; they are not corrected by hand.

In [ ]:
EXPECTED_COLUMNS = [
    "chat_id", "msg_idx", "username", "datetime_text", "text",
    "category", "speaker_role"
]
VALID_CATEGORIES = {"200", "600", "900"}

assert list(messages.columns) == EXPECTED_COLUMNS
assert messages["category"].isin(VALID_CATEGORIES).all(), "Unexpected stage code"
assert not messages.duplicated(["chat_id", "msg_idx"]).any(), "Duplicate chat/message index"
assert messages["speaker_role"].eq("predator").all(), "Only predator-coded rows should remain"

messages["category"] = messages["category"].astype("string")


def sequence_diagnostics(group: pd.DataFrame) -> pd.Series:
    sequence = group.sort_values("msg_idx")["category"].astype(int).tolist()
    numeric = np.array(sequence, dtype=int)
    transitions = list(zip(numeric[:-1], numeric[1:]))
    reversals = sum(next_value < previous_value for previous_value, next_value in transitions)
    return pd.Series({
        "sequence": " -> ".join(map(str, sequence)),
        "reversal_count": reversals,
        "reached_600": 600 in numeric,
        "reached_900": 900 in numeric,
    })


sequence_summary = (
    messages.groupby("chat_id", sort=False, group_keys=False)
    .apply(sequence_diagnostics)
    .reset_index()
)

chat_summary = chat_summary.merge(sequence_summary, on="chat_id", how="left")

display(chat_summary)
display(messages["category"].value_counts().sort_index().rename_axis("category").to_frame("count"))

The inventory check below documents the mismatch between the project's plan and the files actually present. The dataset is not padded or assumed to contain an unavailable 57th file.

In [ ]:
print("Available coded chat XML files:", chat_summary.shape[0])
print("Stage 1 classes present:", sorted(messages["category"].unique().tolist()))
print("Expected grouped unit: chat_id")

stage_distribution = (
    messages.groupby("category", observed=True)
    .size()
    .rename("count")
    .reset_index()
)

messages.to_csv(OUTPUT_DIR / "coded_stage_messages.csv", index=False)
chat_summary.to_csv(OUTPUT_DIR / "coded_chat_summary.csv", index=False)
stage_distribution.to_csv(OUTPUT_DIR / "coded_stage_distribution.csv", index=False)

print("Saved:", OUTPUT_DIR / "coded_stage_messages.csv")
print("Saved:", OUTPUT_DIR / "coded_chat_summary.csv")
print("Saved:", OUTPUT_DIR / "coded_stage_distribution.csv")

## Leakage and small-sample notes

* Cross-validation must group on `chat_id`; message-level rows from the same conversation cannot be split across train and test folds.
* With 56 files, report repeated grouped results and uncertainty intervals rather than a single optimistic split.
* Uncoded victim posts are not Stage 1 targets.
* The parser should be rerun if a 57th coded XML is added; the notebook must report, not assume, the actual inventory count.